# Task A Run 36 -- Gemma-4-12B, 3 epochs, two-seed full-data submission

This is the released-test final fit for the winning Task A Run 29 configuration. It trains two independent Gemma-4-12B QLoRA classifiers (seeds 42 and 43) for 3 epochs on all 7,193 labelled Task A rows, averages their test probabilities, and writes one CodaBench-ready ZIP.

Run 29's matched holdout result was **0.8610 macro-F1** for the two-seed 3-epoch blend. The rejected four-epoch blend scored 0.8452. This notebook is a full-data fit, so it does not report a local test score: the released test labels are unavailable.

Output: `/kaggle/working/a36_outputs/submissions/RECOMMENDED_a36_gemma_3ep_2seeds.zip`. The ZIP contains exactly `predictions.csv`.

**Settings:** Kaggle Internet on, GPU T4 x2, Save Version → Save & Run All. A Hugging Face token is not normally required for Gemma-4, but a Kaggle secret named `HF_TOKEN` is read if present.

In [ ]:
import json, os, pathlib, re, subprocess, sys, time
from fnmatch import fnmatch

T_START = time.time()
HARD_STOP = T_START + 11 * 3600
REPO = 'https://github.com/robinpnalex/Hastika-ICON2026.git'
BRANCH = 'task-b'
WORK = '/tmp/hastika'

def git(*args):
    return subprocess.run(['git', '-C', WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

remote = subprocess.run(['git', 'ls-remote', '--exit-code', REPO, f'refs/heads/{BRANCH}'],
                        check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
if os.path.isdir(WORK + '/.git'):
    git('fetch', '-q', '--depth', '1', 'origin', BRANCH)
    git('reset', '-q', '--hard', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, '--depth', '1', REPO, WORK], check=True)
head = git('rev-parse', 'HEAD')
assert head == remote, f'clone is at {head[:8]} but {BRANCH} is at {remote[:8]}'
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
print('repo:', git('log', '-1', '--oneline'))

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf \"transformers>=4.45,<6\"', shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), 'select GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('gpus:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])

OUT = pathlib.Path('/kaggle/working/a36_outputs')
RUNS, LOGS, SUB = OUT / 'runs', OUT / 'logs', OUT / 'submissions'
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = '/tmp/hf'
os.environ['HF_HUB_CACHE'] = HF_CACHE

from hastika.common.gpu_queue import run_queue

TASK = 'a'
MODEL = 'google/gemma-4-12B'
CLASSES = ['Non-Hate', 'Hate']
LABEL_COL = 'Label'
SPLIT = pathlib.Path(WORK) / 'data/derived/task_a_combined_holdout'
train = pd.read_csv(SPLIT / 'all.csv')
test = pd.read_csv(pathlib.Path(WORK) / 'data/raw/hastika_binary_test.csv')
TEST_CSV = pathlib.Path('/tmp/task_a_test.csv')
test[['id', 'Comment']].to_csv(TEST_CSV, index=False)
assert len(train) == 7193, len(train)
assert len(test) == 806, len(test)
print(f'full training rows: {len(train)}; test rows: {len(test)}')


In [ ]:
LLM_PY = '/tmp/llmenv/bin/python'
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', '/tmp/llmenv'], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q \"transformers>=5.12,<6\" \"peft>=0.17\" \"bitsandbytes>=0.46\" \"accelerate>=1.0\" emoji ftfy sentencepiece protobuf', shell=True, check=True)
subprocess.run([LLM_PY, '-c', "import transformers, peft, bitsandbytes; print('LLM env:', transformers.__version__, peft.__version__)"], check=True)

def download():
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get('HF_TOKEN') or None,
                      allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', 'tokenizer*', '*.jinja'])

def job(seed):
    name = f'full_s{seed}'
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} '
           f'--train {SPLIT / "all.csv"} --predict {TEST_CSV} --out {RUNS / name} '
           f'--seed {seed} --epochs 3 --prompt short --lr 1e-4 --bs 8 --grad-accum 2 '
           f'--eval-bs 16 --cache {HF_CACHE}')
    return {'name': name, 'cmd': cmd, 'log': str(LOGS / f'{name}.log')}

def load_probs(seed):
    run = RUNS / f'full_s{seed}'
    probs_path = run / 'task_a_test_probs.npy'
    ids_path = run / 'task_a_test_ids.csv'
    # llm_classifier uses the input stem; support the actual task_a_test stem too.
    candidates = [(run / f'{TEST_CSV.stem}_probs.npy', run / f'{TEST_CSV.stem}_ids.csv'),
                 (probs_path, ids_path), (run / 'hastika_binary_test_probs.npy', run / 'hastika_binary_test_ids.csv')]
    for p, ids in candidates:
        if p.exists() and ids.exists():
            got = pd.read_csv(ids)['id'].tolist()
            if got == test['id'].tolist():
                probs = np.load(p)
                assert probs.shape == (len(test), len(CLASSES)), (seed, probs.shape)
                return probs
    return None

def run_jobs(jobs):
    registry = {j['name']: j for j in jobs}
    retried = set()
    def on_done(name, code):
        if not code or name in retried:
            return []
        job0 = registry[name]
        log = pathlib.Path(job0['log']).read_text(errors='replace') if pathlib.Path(job0['log']).exists() else ''
        retry = None
        m = re.search(r'--seed (\d+)', job0['cmd'])
        if code == 4 and m:
            retry = {**job0, 'cmd': job0['cmd'] + f" --seed {int(m.group(1)) + 1000}"}
            print(f'{name}: collapse detected; retrying with alternate seed')
        elif 'OutOfMemoryError' in log:
            retry = {**job0, 'cmd': job0['cmd'] + ' --bs 4 --grad-accum 4 --eval-bs 8'}
            print(f'{name}: OOM detected; retrying with smaller micro-batches')
        if retry is None:
            return []
        retried.add(name)
        registry[name] = retry
        return [retry]
    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=on_done, stop_at=HARD_STOP)

def write_zip(probs):
    import zipfile
    pred = pd.DataFrame({'id': test['id'], 'label': [CLASSES[i] for i in probs.argmax(1)]})
    assert pred['id'].is_unique and len(pred) == 806 and set(pred['label']) <= set(CLASSES)
    np.save(SUB / 'a36_gemma_3ep_2seeds_test_probs.npy', probs)
    csv = SUB / 'predictions_a36_gemma_3ep_2seeds.csv'
    pred.to_csv(csv, index=False)
    out = SUB / 'RECOMMENDED_a36_gemma_3ep_2seeds.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(csv, 'predictions.csv')
    print('created:', out)
    print(pred['label'].value_counts(normalize=True).rename('predicted_share'))
    return out

download()
jobs = [job(seed) for seed in (42, 43)]
print('queued:', [j['name'] for j in jobs])
codes = run_jobs(jobs)
members = [p for seed in (42, 43) if (p := load_probs(seed)) is not None]
print('completed members:', len(members), 'exit codes:', codes)
assert len(members) == 2, 'both seed predictions are required for the intended two-seed submission'
probs = np.mean(members, axis=0)
write_zip(probs)
json.dump({'commit': head, 'task': 'a', 'epochs': 3, 'prompt': 'short', 'tapt': False,
           'lr': 1e-4, 'seeds': [42, 43], 'rows': len(train), 'test_rows': len(test),
           'exit_codes': codes, 'elapsed_hours': (time.time() - T_START) / 3600},
          open(OUT / 'result.json', 'w'), indent=2)
print(f'finished in {(time.time() - T_START) / 3600:.2f} h')
